# Job A — Feature-based Real-IAD benchmark (Colab)

Runs PatchCore + PaDiM + SubspaceAD on 30 Real-IAD categories, camera C1 only.

**Before running:**
- Runtime -> Change runtime type -> GPU (T4/V100/A100).
- `Real-IAD_dataset/realiad_1024/*.zip` must be on Drive (zipped, one per category).

**Outputs:** synced to `/content/drive/MyDrive/thesis_runs/jobA/` per-category. Safe to
interrupt and restart — finished categories get a `<category>.done` marker and are skipped on resume.

In [ ]:
# 1. Mount Drive and verify the Real-IAD zips are visible.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
ZIPS_DIR = '/content/drive/MyDrive/Real-IAD_dataset/realiad_1024'
zips = sorted(glob.glob(os.path.join(ZIPS_DIR, '*.zip')))
print(f'Found {len(zips)} category zips under {ZIPS_DIR}')
for z in zips[:5]:
    print('  -', os.path.basename(z))
assert len(zips) > 0, 'No zips found — check ZIPS_DIR matches your Drive layout.'

In [ ]:
# 2. Clone (or pull) the thesis repo into /content/.
# Replace <your-username>/<your-repo> with the repo you pushed to.
REPO_URL = 'https://github.com/<your-username>/Real-time-visual-defect-detection.git'
REPO_DIR = '/content/Real-time-visual-defect-detection'
BRANCH   = 'main'

import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--all'])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])

print('Repo ready at', REPO_DIR)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 3. Install Python dependencies for the three models.
# Colab ships with torch + cv2; we add anomalib, lightning, transformers, FrEIA-less stack.
!pip install -q anomalib==1.1.* lightning==2.3.* transformers==4.44.* scikit-learn timm
!pip install -q -e /content/Real-time-visual-defect-detection --no-deps

import torch, anomalib, lightning, transformers
print('torch       ', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('anomalib    ', anomalib.__version__)
print('lightning   ', lightning.__version__)
print('transformers', transformers.__version__)

In [ ]:
# 4. Launch the category loop. Streams logs live; safe to interrupt and rerun.
import os
os.environ['ZIPS_DIR']    = '/content/drive/MyDrive/Real-IAD_dataset/realiad_1024'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/thesis_runs/jobA'
os.environ['WORK_DIR']    = '/content/work'
os.environ['REPO_DIR']    = '/content/Real-time-visual-defect-detection'
os.environ['CONFIG']      = '/content/Real-time-visual-defect-detection/src/benchmark_AD/configs/colab_featurebased.yaml'

!bash /content/Real-time-visual-defect-detection/scripts/run_jobA_colab.sh 2>&1 | tee -a /content/drive/MyDrive/thesis_runs/jobA/run.log

## Troubleshooting

- **`No Real-IAD images found`**: the zip did not unpack with the expected `OK/` + `NG/<defect>/S000X/` layout. Inspect one extracted category in `/content/work/<category>/` and share the `ls` output.
- **`CUDA out of memory` on SubspaceAD**: lower `subspacead.batch_size` in the config (default 4) to 2.
- **Session disconnected mid-run**: reopen the notebook, re-run cells 1-3, then rerun cell 4 — completed categories are skipped via `.done` markers.
- **Drive I/O stalls**: check `Runtime -> Manage sessions`; if Drive is throttled, wait a few minutes or reconnect.